In [1]:
from pathlib import Path  # Handles Windows/project paths safely.

import zipfile  # Reads zip files without extracting the whole archive.

import shutil  # Copies one nested zip temporarily so Python can inspect it.

import re  # Helps parse setting names and values from text.

import pandas as pd  # Builds summary tables and saves Excel files.

from IPython.display import display  # Displays notebook previews.

In [ ]:
project_root = Path.cwd().parent  # Moves from Z:\Research-analysis\notebook to Z:\Research-analysis.

raw_folder = project_root / "data" / "raw" / "LUTO2-Scenario-Run-2"  # Folder containing the transfer zip.

processed_folder = project_root / "data" / "processed"  # Folder where final outputs will be saved.

notebook_name = "10_model_run_settings_summary"  # Output folder will use this notebook name.

output_folder = processed_folder / notebook_name  # Creates data/processed/10_model_run_settings_summary.

output_folder.mkdir(parents=True, exist_ok=True)  # Creates the output folder if missing.

transfer_zip = raw_folder / "transfer_3573951_files_89a7061e.zip"  # Main transfer zip file.

model_settings_index = processed_folder / "09_transfer_file_inventory" / "Model run settings.xlsx"  # Excel file listing the 16 model setting file locations.

print("Transfer zip:", transfer_zip)  # Confirms the main zip path.

print("Settings index:", model_settings_index)  # Confirms the Excel file with model setting locations.

print("Transfer zip exists:", transfer_zip.exists())  # Checks whether the transfer zip is found.

print("Settings index exists:", model_settings_index.exists())  # Checks whether the Excel file is found.

In [ ]:
settings_locations = pd.read_excel(model_settings_index)  # Reads the Excel file listing the 16 model setting files.

settings_locations = settings_locations.sort_values("Run Sheet").reset_index(drop=True)  # Sorts Run_G0001 to Run_G0016.

display(settings_locations)  # Shows the file-location table for checking.

In [4]:
def clean_path(path):  # Standardizes zip paths.
    return str(path).replace("\\", "/").strip()  # Converts backslashes to forward slashes.


def find_zip_member(zip_obj, wanted_path):  # Finds a file inside a zip by exact or ending match.
    wanted_path = clean_path(wanted_path)  # Cleans the target path.

    for info in zip_obj.infolist():  # Loops through zip members.
        if clean_path(info.filename) == wanted_path:  # Checks exact path match.
            return info  # Returns the matched file.

    for info in zip_obj.infolist():  # Loops again for flexible matching.
        if clean_path(info.filename).endswith("/" + wanted_path):  # Checks suffix match.
            return info  # Returns the matched file.

    raise FileNotFoundError(f"Could not find {wanted_path}")  # Stops with a clear error.


def decode_text(raw_bytes):  # Converts file bytes into readable text.
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:  # Tries common encodings.
        try:
            return raw_bytes.decode(encoding)  # Returns decoded text if successful.
        except UnicodeDecodeError:
            pass  # Tries the next encoding.
    return raw_bytes.decode("latin-1", errors="replace")  # Fallback decoding.


def parse_settings_text(run_name, text, source_file):  # Converts settings text into rows.
    rows = []  # Stores parsed settings.

    for line_no, raw_line in enumerate(text.splitlines(), start=1):  # Reads text line by line.
        line = raw_line.strip()  # Removes extra spaces.

        if not line or line.startswith(("#", ";", "//")):  # Skips blank/comment lines.
            continue

        if "=" in line:  # Handles setting = value format.
            setting, value = line.split("=", 1)
        elif ":" in line:  # Handles setting: value format.
            setting, value = line.split(":", 1)
        else:  # Handles lines without obvious separators.
            setting, value = f"line_{line_no:04d}", line

        setting = re.sub(r"\s+", " ", setting.strip())  # Cleans setting name.
        value = value.strip()  # Cleans setting value.

        rows.append({"Run No.": run_name, "Setting": setting, "Value": value, "Source File": source_file})  # Adds one setting row.

    return rows  # Returns parsed settings.

In [ ]:
all_settings = []  # Stores all parsed setting rows.

errors = []  # Stores any files that could not be read.

temp_folder = output_folder / "_temporary_nested_zips"  # Temporary folder for one nested zip at a time.

temp_folder.mkdir(exist_ok=True)  # Creates the temporary folder.

with zipfile.ZipFile(transfer_zip, "r") as main_zip:  # Opens the main transfer zip without extracting it.
    for _, row in settings_locations.iterrows():  # Loops through each run setting file.
        run_name = row["Run Sheet"]  # Gets Run_G0001, Run_G0002, etc.

        file_location = clean_path(row["File Location"])  # Gets full nested location from Excel.

        outer_zip_path, inner_txt_path = [x.strip() for x in file_location.split("::", 1)]  # Splits archive path and text path.

        print("Processing", run_name)  # Shows progress.

        try:
            outer_info = find_zip_member(main_zip, outer_zip_path)  # Finds Run_Archive.zip inside the main zip.

            temp_zip_path = temp_folder / f"{run_name}_Run_Archive_temp.zip"  # Temporary copy of the nested zip.

            with main_zip.open(outer_info) as source, open(temp_zip_path, "wb") as target:  # Opens nested zip stream and temp file.
                shutil.copyfileobj(source, target, length=1024 * 1024 * 16)  # Copies in 16 MB chunks.

            with zipfile.ZipFile(temp_zip_path, "r") as run_zip:  # Opens the temporary Run_Archive.zip.
                inner_info = find_zip_member(run_zip, inner_txt_path)  # Finds model_run_settings.txt inside it.
                text = decode_text(run_zip.read(inner_info))  # Reads and decodes the settings text.

            all_settings.extend(parse_settings_text(run_name, text, file_location))  # Adds parsed settings to the master list.

            temp_zip_path.unlink(missing_ok=True)  # Deletes the temporary nested zip.

        except Exception as error:
            errors.append({"Run No.": run_name, "File Location": file_location, "Error": str(error)})  # Records failure.
            print("Error:", run_name, error)  # Prints the error.

In [ ]:
settings_long = pd.DataFrame(all_settings)  # Creates a long table: one setting per row.

settings_wide = settings_long.pivot_table(  # Creates one row per run and one column per setting.
    index="Run No.",
    columns="Setting",
    values="Value",
    aggfunc=lambda values: " | ".join(pd.unique(values.astype(str)))
).reset_index()

excel_output = output_folder / "model_run_settings_summary.xlsx"  # Final Excel output path.

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:  # Opens Excel writer.
    settings_wide.to_excel(writer, sheet_name="Settings_Wide", index=False)  # Saves screenshot-style table.
    settings_long.to_excel(writer, sheet_name="Settings_Long", index=False)  # Saves audit-friendly long table.
    settings_locations.to_excel(writer, sheet_name="Source_Files", index=False)  # Saves source file list.
    pd.DataFrame(errors).to_excel(writer, sheet_name="Errors", index=False)  # Saves read errors if any.

print("Saved:", excel_output)  # Prints the final file path.

display(settings_wide)  # Shows the final wide table preview.